# LLM File Chat

Load any file, send it to an LLM with a prompt, and save the response.

In [4]:
!pip install generic-llm-api-client

  Using cached colab-1.13.5.tar.gz (567 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'error'


  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> [7 lines of output]
      C:\Users\sorin\AppData\Local\Temp\pip-build-env-ppjaa6sz\overlay\Lib\site-packages\setuptools\_distutils\dist.py:287: UserWarning: Unknown distribution option: 'tests_require'
        warnings.warn(msg)
      C:\Users\sorin\AppData\Local\Temp\pip-build-env-ppjaa6sz\overlay\Lib\site-packages\setuptools\_distutils\dist.py:287: UserWarning: Unknown distribution option: 'test_suite'
        warnings.warn(msg)
      error in colab setup command: 'install_requires' must be a string or iterable of strings containing valid project/version requirement specifiers; Expected comma (within version specifier), semicolon (after version specifier) or end
          pytz>=2011n
              ~~~~~~^
      [end of output]
  
  note: This error originates from a subprocess, and is likely not a problem with pip.

[notice] A new release of pip is availab

## Configuration

In [ ]:
from google.colab import files
uploaded = files.upload()
print("Uploaded files:", list(uploaded.keys()))

# --- LLM settings ---
LLM_PROVIDER  = "openai"       # "openai", "anthropic", "google", "mistral", "deepseek", "openrouter"
LLM_API_KEY   = "sk-..."
LLM_MODEL     = "gpt-4o"

# --- Files ---
INPUT_FILE    = list(uploaded.keys())[0]   # path to the file you want to send
OUTPUT_FILE   = "output.txt"   # path where the LLM response will be saved

## Set up LLM client

In [ ]:
from ai_client import create_ai_client

llm = create_ai_client(LLM_PROVIDER, api_key=LLM_API_KEY)
print(f"LLM client ready ({LLM_PROVIDER} / {LLM_MODEL})")

## Load file

In [ ]:
import pathlib

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".gif", ".webp"}
TEXT_EXTENSIONS  = {".txt", ".xml", ".json", ".csv", ".md", ".html", ".htm"}

suffix = pathlib.Path(INPUT_FILE).suffix.lower()

if suffix in IMAGE_EXTENSIONS:
    file_mode = "image"
elif suffix == ".pdf":
    file_mode = "pdf"
elif suffix in TEXT_EXTENSIONS:
    file_mode = "text"
    with open(INPUT_FILE, encoding="utf-8") as f:
        file_content = f.read()
else:
    # fallback: try reading as text
    file_mode = "text"
    with open(INPUT_FILE, encoding="utf-8") as f:
        file_content = f.read()

print(f"Loaded {INPUT_FILE!r}  →  mode: {file_mode}")
if file_mode == "text":
    print(f"({len(file_content):,} characters)")
    print("--- Preview ---")
    print(file_content[:800])

## Prompt

Edit `SYSTEM_PROMPT` and `USER_PROMPT` for your task.

In [ ]:
SYSTEM_PROMPT = """\
You are a helpful assistant.
"""

# For text files the content is embedded directly in the prompt.
# For images/PDFs the file is passed via the dedicated parameter.
if file_mode == "text":
    USER_PROMPT = f"Here is the file content:\n\n{file_content}"
    kwargs = {}
else:
    USER_PROMPT = "Please analyse the attached file."
    kwargs = {"images": [INPUT_FILE]} if file_mode == "image" else {"files": [INPUT_FILE]}

print("Sending to LLM…")
response = llm.prompt(LLM_MODEL, USER_PROMPT, system_prompt=SYSTEM_PROMPT, **kwargs)
answer = response.text

print(f"Done ({response.usage.total_tokens:,} tokens, {response.duration:.1f}s)")
print("--- Response preview ---")
print(answer[:800])

## Save response

In [ ]:
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    f.write(answer)

print(f"Saved to {OUTPUT_FILE!r}")